# Step 11 — Final preliminary cell-type annotation and composition plots

This notebook consolidates the completed Step 10 inclusive rescue into one
preliminary annotation for every sample.

The **primary analysis definition** is:

```text
T lineage:
    rescue_T_primary

Treg:
    rescue_Treg_supported

CD4/CD8:
    rescue_T_subtype_primary

Non-T lineage:
    fallback_level1_lineage
```

The notebook does not rerun ResolVI, pyUCell, clustering, integration, or
threshold optimization.

## Final primary cell types

```text
Treg
CD4+ T
CD8+ T
Other T
NK
B/plasma
Monocyte/macrophage
Fibroblast/stromal
Endothelial
Tumor
Other/unresolved
```

`Treg` has priority over CD4/CD8. The final Treg label uses the **supported**
Step 10 tier, while confidence is retained separately.

## Confidence and provenance columns

The full annotation keeps nested evidence rather than collapsing everything
into one label:

```text
prelim_T_confidence_tier
    strict
    primary_rescue
    exploratory_only
    not_T

prelim_T_confidence_rank
    3, 2, 1, 0

prelim_Treg_confidence_tier
    high_confidence
    supported
    exploratory_only
    not_Treg

prelim_Treg_confidence_rank
    3, 2, 1, 0
```

The original Step 08/10 evidence columns are preserved, including:

```text
rescue_T_mixing_status
rescue_raw_T_coherent
rescue_reference_both_T
rescue_raw_FOXP3_detected
rescue_raw_Treg_anchor
rescue_Treg_high_confidence
rescue_Treg_supported
rescue_Treg_exploratory
```

## Two parallel annotations

```text
prelim_cell_type_primary
    uses primary T + supported Treg
    recommended for the preliminary deliverable

prelim_cell_type_exploratory
    uses exploratory T + exploratory Treg
    retained for sensitivity analysis
```

Plots default to the primary annotation.

## Main figures

For each cancer type:

1. patient-level total stacked cell counts;
2. paired Screen/C2D15 stacked counts within each patient;
3. 100% stacked cell-type percentages for every sample;
4. 100% stacked T-cell-subtype percentages among primary T cells.

For each individual patient:

```text
Screen versus C2D15 paired stacked count chart
```

## Matrix convention

The output H5AD keeps the input Step 10 matrix unchanged:

```python
adata.X
# original sparse raw count matrix
```

Corrected-expression-derived UCell scores and rescue evidence remain in
`.obs`. The dense corrected matrix is not duplicated.

In [1]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import json
import re
import traceback
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

anndata: 0.12.19
numpy: 1.26.4
pandas: 2.3.3


In [2]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

RESCUE_ROOT = (
    PIPELINE_ROOT
    / "10_final_inclusive_T_Treg_rescue"
)
OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "11_preliminary_cell_type_annotation"
)
FIGURE_ROOT = (
    OUTPUT_ROOT
    / "figures"
)
TABLE_ROOT = (
    OUTPUT_ROOT
    / "tables"
)
PATIENT_FIGURE_ROOT = (
    FIGURE_ROOT
    / "individual_patients"
)

for path in (
    OUTPUT_ROOT,
    FIGURE_ROOT,
    TABLE_ROOT,
    PATIENT_FIGURE_ROOT,
):
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen",
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15",
    },
}

SECTION_NAMES = list(
    SAMPLE_INFO
)
CANCER_TYPE_ORDER = [
    "melanoma",
    "NSCLC",
    "colon_cancer",
]
BIOPSY_STAGE_ORDER = [
    "Screen",
    "C2D15",
]

# The main deliverable uses the Step 10 primary T and supported-Treg tiers.
PRIMARY_T_COLUMN = (
    "rescue_T_primary"
)
PRIMARY_TREG_COLUMN = (
    "rescue_Treg_supported"
)

EXPLORATORY_T_COLUMN = (
    "rescue_T_exploratory"
)
EXPLORATORY_TREG_COLUMN = (
    "rescue_Treg_exploratory"
)

# Detailed annotation retained in H5AD.
DETAILED_CELL_TYPE_ORDER = [
    "Tumor",
    "Fibroblast/stromal",
    "Endothelial",
    "Monocyte/macrophage",
    "B/plasma",
    "NK",
    "Other/unresolved",
    "Other T",
    "CD4+ T",
    "CD8+ T",
    "Treg",
]

# Plot-level annotation collapses ambiguous and unspecified T cells into Other T.
PLOT_CELL_TYPE_ORDER = DETAILED_CELL_TYPE_ORDER.copy()

T_SUBTYPE_ORDER = [
    "Other T",
    "CD4+ T",
    "CD8+ T",
    "Treg",
]

WRITE_ANNOTATED_H5AD = True
WRITE_PRIMARY_T_ONLY_H5AD = True
WRITE_CELL_METADATA_PARQUET = True
WRITE_SPATIAL_PARQUET = True
VALIDATE_WRITTEN_H5AD = True

H5AD_COMPRESSION = "lzf"
PLOT_DPI = 500
CONTINUE_ON_ERROR = True

PIPELINE_VERSION = (
    "2026-08-02-preliminary-cell-type-annotation-v1"
)

print("Step 10 source:", RESCUE_ROOT)
print("Output root:", OUTPUT_ROOT)

Step 10 source: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation


In [3]:
# ---------------------------------------------------------------------
# Paths and general helpers
# ---------------------------------------------------------------------
def sample_paths(
    sample: str,
) -> dict[str, Path]:
    source = (
        RESCUE_ROOT
        / sample
        / f"{sample}_inclusive_rescue_annotated.h5ad"
    )
    out = (
        OUTPUT_ROOT
        / sample
    )
    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "source": source,
        "out": out,
        "annotated": (
            out
            / f"{sample}_preliminary_cell_type_annotated.h5ad"
        ),
        "primary_T": (
            out
            / f"{sample}_preliminary_primary_Tcells.h5ad"
        ),
        "metadata": (
            out
            / f"{sample}_preliminary_cell_type_metadata.parquet"
        ),
        "spatial": (
            out
            / f"{sample}_preliminary_cell_type_spatial.parquet"
        ),
        "counts": (
            out
            / f"{sample}_preliminary_cell_type_counts.csv"
        ),
        "confidence": (
            out
            / f"{sample}_T_Treg_confidence_counts.csv"
        ),
        "summary": (
            out
            / f"{sample}_preliminary_annotation_summary.json"
        ),
    }


def safe_slug(
    value: str,
) -> str:
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


def write_json(
    payload,
    path: str | Path,
) -> None:
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    temporary.replace(
        path
    )


def choose_spatial_key(
    adata: ad.AnnData,
) -> str | None:
    for key in (
        "X_spatial",
        "spatial",
        "spatial_fullres",
    ):
        if key not in adata.obsm:
            continue

        value = np.asarray(
            adata.obsm[key]
        )
        if (
            value.ndim == 2
            and value.shape[0]
            == adata.n_obs
            and value.shape[1]
            >= 2
        ):
            return key

    return None


def bool_obs(
    adata: ad.AnnData,
    column: str,
) -> np.ndarray:
    if column not in adata.obs.columns:
        raise KeyError(
            f"Missing required boolean column: {column}"
        )
    return (
        adata.obs[column]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )


def text_obs(
    adata: ad.AnnData,
    column: str,
    default: str = "",
) -> np.ndarray:
    if column not in adata.obs.columns:
        return np.repeat(
            default,
            adata.n_obs,
        )
    return (
        adata.obs[column]
        .astype("string")
        .fillna(default)
        .astype(str)
        .to_numpy(
            dtype=object
        )
    )

In [4]:
# ---------------------------------------------------------------------
# Portable H5AD writer
# ---------------------------------------------------------------------
def _is_nullable_string_series(
    series: pd.Series,
) -> bool:
    return (
        isinstance(
            series.dtype,
            pd.StringDtype,
        )
        or type(
            series.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
        or str(
            series.dtype
        ) == "str"
        or str(
            series.dtype
        ).startswith(
            "string"
        )
    )


def _legacy_string_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    for column in output.columns:
        series = output[column]

        if _is_nullable_string_series(
            series
        ):
            output[column] = (
                series
                .fillna("")
                .astype(str)
                .astype(object)
            )
            continue

        if pd.api.types.is_object_dtype(
            series.dtype
        ):
            def _safe_text(value):
                if (
                    value is None
                    or value is pd.NA
                ):
                    return ""
                if isinstance(
                    value,
                    bytes,
                ):
                    return value.decode(
                        "utf-8",
                        errors="replace",
                    )
                if isinstance(
                    value,
                    str,
                ):
                    return value
                if isinstance(
                    value,
                    np.generic,
                ):
                    value = value.item()
                if isinstance(
                    value,
                    (
                        dict,
                        list,
                        tuple,
                        set,
                        np.ndarray,
                        Path,
                    ),
                ):
                    return json.dumps(
                        value,
                        default=str,
                        sort_keys=True,
                    )
                return str(
                    value
                )

            output[column] = (
                series
                .map(
                    _safe_text
                )
                .astype(object)
            )

    if (
        str(
            output.index.dtype
        ) == "str"
        or str(
            output.index.dtype
        ).startswith(
            "string"
        )
        or type(
            output.index.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
    ):
        index_name = (
            output.index.name
        )
        output.index = pd.Index(
            pd.Series(
                output.index,
                dtype="string",
            )
            .fillna("")
            .astype(str)
            .to_numpy(
                dtype=object
            ),
            name=index_name,
        )

    return output


def safe_write_h5ad(
    adata_object: ad.AnnData,
    filename: str | Path,
    *,
    compression: str = "lzf",
) -> None:
    filename = Path(
        filename
    )
    filename.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    adata_object.obs = (
        _legacy_string_frame(
            adata_object.obs
        )
    )
    adata_object.var = (
        _legacy_string_frame(
            adata_object.var
        )
    )

    temporary = filename.with_name(
        f"{filename.stem}.tmp{filename.suffix}"
    )
    if temporary.exists():
        temporary.unlink()

    with ad.settings.override(
        allow_write_nullable_strings=False
    ):
        adata_object.write_h5ad(
            temporary,
            compression=compression,
            convert_strings_to_categoricals=False,
        )

    temporary.replace(
        filename
    )
    print("Saved:", filename)

In [5]:
# ---------------------------------------------------------------------
# Cell-type construction
# ---------------------------------------------------------------------
BROAD_LINEAGE_MAP = {
    "Tumor": "Tumor",
    "Endothelial": "Endothelial",
    "Monocyte_macrophage": "Monocyte/macrophage",
    "Fibroblast": "Fibroblast/stromal",
    "B_plasma": "B/plasma",
    "NK": "NK",
    "Other_unresolved": "Other/unresolved",
    "T_lineage": "Other T",
}

T_SUBTYPE_MAP = {
    "Tcell:CD4+": "CD4+ T",
    "Tcell:CD8+": "CD8+ T",
    "Tcell:CD4_CD8_ambiguous": "Other T",
    "Tcell:unspecified": "Other T",
    "Tcell:Treg_supported": "Treg",
    "Tcell:Treg_high_confidence": "Treg",
    "Tcell:Treg_exploratory": "Treg",
    "": "Other T",
}


def map_broad_lineage(
    broad_values: np.ndarray,
) -> np.ndarray:
    return np.asarray(
        [
            BROAD_LINEAGE_MAP.get(
                str(value),
                "Other/unresolved",
            )
            for value in broad_values
        ],
        dtype=object,
    )


def map_T_subtype(
    subtype_values: np.ndarray,
) -> np.ndarray:
    return np.asarray(
        [
            T_SUBTYPE_MAP.get(
                str(value),
                "Other T",
            )
            for value in subtype_values
        ],
        dtype=object,
    )


def add_preliminary_annotation(
    adata: ad.AnnData,
    sample: str,
) -> dict:
    required_columns = {
        "fallback_level1_lineage",
        "fallback_T_lineage",
        "rescue_T_tier",
        PRIMARY_T_COLUMN,
        EXPLORATORY_T_COLUMN,
        PRIMARY_TREG_COLUMN,
        EXPLORATORY_TREG_COLUMN,
        "rescue_Treg_high_confidence",
        "rescue_T_subtype_primary",
        "rescue_T_subtype_exploratory",
        "rescue_T_mixing_status",
    }
    missing = sorted(
        required_columns
        - set(
            adata.obs.columns
        )
    )
    if missing:
        raise KeyError(
            f"{sample}: Step 10 object is missing required columns: {missing}"
        )

    strict_T = bool_obs(
        adata,
        "fallback_T_lineage",
    )
    primary_T = bool_obs(
        adata,
        PRIMARY_T_COLUMN,
    )
    exploratory_T = bool_obs(
        adata,
        EXPLORATORY_T_COLUMN,
    )

    Treg_high = bool_obs(
        adata,
        "rescue_Treg_high_confidence",
    )
    Treg_supported = bool_obs(
        adata,
        PRIMARY_TREG_COLUMN,
    )
    Treg_exploratory = bool_obs(
        adata,
        EXPLORATORY_TREG_COLUMN,
    )

    # Nested-tier validation.
    if np.any(
        strict_T
        & ~primary_T
    ):
        raise ValueError(
            f"{sample}: strict T is not a subset of primary T."
        )
    if np.any(
        primary_T
        & ~exploratory_T
    ):
        raise ValueError(
            f"{sample}: primary T is not a subset of exploratory T."
        )
    if np.any(
        Treg_high
        & ~Treg_supported
    ):
        raise ValueError(
            f"{sample}: high-confidence Treg is not a subset of supported Treg."
        )
    if np.any(
        Treg_supported
        & ~Treg_exploratory
    ):
        raise ValueError(
            f"{sample}: supported Treg is not a subset of exploratory Treg."
        )
    if np.any(
        Treg_supported
        & ~primary_T
    ):
        raise ValueError(
            f"{sample}: supported Treg is not a subset of primary T."
        )

    broad = text_obs(
        adata,
        "fallback_level1_lineage",
        default="Other_unresolved",
    )
    broad_labels = map_broad_lineage(
        broad
    )

    primary_subtype_source = text_obs(
        adata,
        "rescue_T_subtype_primary",
    )
    exploratory_subtype_source = text_obs(
        adata,
        "rescue_T_subtype_exploratory",
    )

    primary_T_labels = map_T_subtype(
        primary_subtype_source
    )
    exploratory_T_labels = map_T_subtype(
        exploratory_subtype_source
    )

    # Explicit Treg priority.
    primary_T_labels[
        Treg_supported
    ] = "Treg"
    exploratory_T_labels[
        Treg_exploratory
    ] = "Treg"

    primary_cell_type = (
        broad_labels.copy()
    )
    primary_cell_type[
        primary_T
    ] = primary_T_labels[
        primary_T
    ]

    exploratory_cell_type = (
        broad_labels.copy()
    )
    exploratory_cell_type[
        exploratory_T
    ] = exploratory_T_labels[
        exploratory_T
    ]

    # Confidence tiers.
    T_confidence = np.full(
        adata.n_obs,
        "not_T",
        dtype=object,
    )
    T_confidence[
        exploratory_T
    ] = "exploratory_only"
    T_confidence[
        primary_T
    ] = "primary_rescue"
    T_confidence[
        strict_T
    ] = "strict"

    T_confidence_rank = np.zeros(
        adata.n_obs,
        dtype=np.int8,
    )
    T_confidence_rank[
        exploratory_T
    ] = 1
    T_confidence_rank[
        primary_T
    ] = 2
    T_confidence_rank[
        strict_T
    ] = 3

    Treg_confidence = np.full(
        adata.n_obs,
        "not_Treg",
        dtype=object,
    )
    Treg_confidence[
        Treg_exploratory
    ] = "exploratory_only"
    Treg_confidence[
        Treg_supported
    ] = "supported"
    Treg_confidence[
        Treg_high
    ] = "high_confidence"

    Treg_confidence_rank = np.zeros(
        adata.n_obs,
        dtype=np.int8,
    )
    Treg_confidence_rank[
        Treg_exploratory
    ] = 1
    Treg_confidence_rank[
        Treg_supported
    ] = 2
    Treg_confidence_rank[
        Treg_high
    ] = 3

    adata.obs[
        "prelim_cell_type_primary"
    ] = pd.Categorical(
        primary_cell_type,
        categories=(
            DETAILED_CELL_TYPE_ORDER
        ),
    )
    adata.obs[
        "prelim_cell_type_exploratory"
    ] = pd.Categorical(
        exploratory_cell_type,
        categories=(
            DETAILED_CELL_TYPE_ORDER
        ),
    )

    adata.obs[
        "prelim_T_subtype_primary"
    ] = pd.Categorical(
        np.where(
            primary_T,
            primary_T_labels,
            "not_T",
        )
    )
    adata.obs[
        "prelim_T_subtype_exploratory"
    ] = pd.Categorical(
        np.where(
            exploratory_T,
            exploratory_T_labels,
            "not_T",
        )
    )

    adata.obs[
        "prelim_T_confidence_tier"
    ] = pd.Categorical(
        T_confidence,
        categories=[
            "strict",
            "primary_rescue",
            "exploratory_only",
            "not_T",
        ],
        ordered=True,
    )
    adata.obs[
        "prelim_T_confidence_rank"
    ] = T_confidence_rank

    adata.obs[
        "prelim_Treg_confidence_tier"
    ] = pd.Categorical(
        Treg_confidence,
        categories=[
            "high_confidence",
            "supported",
            "exploratory_only",
            "not_Treg",
        ],
        ordered=True,
    )
    adata.obs[
        "prelim_Treg_confidence_rank"
    ] = Treg_confidence_rank

    adata.obs[
        "prelim_T_primary"
    ] = primary_T
    adata.obs[
        "prelim_T_exploratory"
    ] = exploratory_T
    adata.obs[
        "prelim_Treg_supported"
    ] = Treg_supported
    adata.obs[
        "prelim_Treg_exploratory"
    ] = Treg_exploratory
    adata.obs[
        "prelim_Treg_high_confidence"
    ] = Treg_high

    # Convenience flags.
    adata.obs[
        "prelim_CD4_T"
    ] = (
        primary_T
        & (
            primary_cell_type
            == "CD4+ T"
        )
    )
    adata.obs[
        "prelim_CD8_T"
    ] = (
        primary_T
        & (
            primary_cell_type
            == "CD8+ T"
        )
    )
    adata.obs[
        "prelim_Treg"
    ] = (
        primary_T
        & (
            primary_cell_type
            == "Treg"
        )
    )

    # Keep human-readable aliases for downstream notebooks.
    adata.obs[
        "cell_type_preliminary"
    ] = adata.obs[
        "prelim_cell_type_primary"
    ].copy()
    adata.obs[
        "T_confidence"
    ] = adata.obs[
        "prelim_T_confidence_tier"
    ].copy()
    adata.obs[
        "Treg_confidence"
    ] = adata.obs[
        "prelim_Treg_confidence_tier"
    ].copy()

    if adata.obs[
        "prelim_cell_type_primary"
    ].isna().any():
        raise ValueError(
            f"{sample}: primary annotation contains missing values."
        )

    counts = (
        adata.obs[
            "prelim_cell_type_primary"
        ]
        .astype(str)
        .value_counts()
        .reindex(
            DETAILED_CELL_TYPE_ORDER,
            fill_value=0,
        )
    )
    if int(
        counts.sum()
    ) != int(
        adata.n_obs
    ):
        raise ValueError(
            f"{sample}: primary cell-type counts do not sum to n_obs."
        )

    return {
        "n_cells": int(
            adata.n_obs
        ),
        "n_strict_T": int(
            strict_T.sum()
        ),
        "n_primary_T": int(
            primary_T.sum()
        ),
        "n_exploratory_T": int(
            exploratory_T.sum()
        ),
        "n_Treg_high_confidence": int(
            Treg_high.sum()
        ),
        "n_Treg_supported": int(
            Treg_supported.sum()
        ),
        "n_Treg_exploratory": int(
            Treg_exploratory.sum()
        ),
        "primary_cell_type_counts": (
            counts.to_dict()
        ),
    }

In [6]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(
    sample: str,
) -> tuple[
    dict,
    pd.DataFrame,
]:
    paths = sample_paths(
        sample
    )

    if not paths[
        "source"
    ].exists():
        raise FileNotFoundError(
            paths[
                "source"
            ]
        )

    print(
        "\n"
        + "=" * 90
    )
    print(
        "Preliminary annotation:",
        sample,
    )
    print(
        "Input:",
        paths[
            "source"
        ],
    )

    adata = ad.read_h5ad(
        paths[
            "source"
        ]
    )
    adata.obs_names = (
        adata.obs_names.astype(str)
    )

    metadata = SAMPLE_INFO[
        sample
    ]
    for column, value in (
        metadata.items()
    ):
        adata.obs[
            column
        ] = value
    adata.obs[
        "sample"
    ] = sample

    annotation_summary = (
        add_preliminary_annotation(
            adata,
            sample,
        )
    )

    # Add compact provenance only.
    adata.uns[
        "preliminary_cell_type_annotation"
    ] = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "source_step10_h5ad": str(
            paths[
                "source"
            ]
        ),
        "primary_T_definition": (
            PRIMARY_T_COLUMN
        ),
        "primary_Treg_definition": (
            PRIMARY_TREG_COLUMN
        ),
        "exploratory_T_definition": (
            EXPLORATORY_T_COLUMN
        ),
        "exploratory_Treg_definition": (
            EXPLORATORY_TREG_COLUMN
        ),
        "primary_cell_type_column": (
            "prelim_cell_type_primary"
        ),
        "exploratory_cell_type_column": (
            "prelim_cell_type_exploratory"
        ),
        "T_confidence_column": (
            "prelim_T_confidence_tier"
        ),
        "Treg_confidence_column": (
            "prelim_Treg_confidence_tier"
        ),
        "CD8B_used": False,
        "matrix_interpretation": (
            "X retains the source Step 10 raw count matrix; "
            "corrected-expression-derived scores remain in obs."
        ),
    }

    # Per-cell metadata sidecar.
    output_columns = [
        column
        for column in adata.obs.columns
        if (
            column.startswith(
                "prelim_"
            )
            or column.startswith(
                "rescue_"
            )
            or column.startswith(
                "fallback_"
            )
            or column
            in {
                "cell_type_preliminary",
                "T_confidence",
                "Treg_confidence",
                "sample",
                "patient",
                "cancer_type",
                "biopsy_stage",
            }
        )
    ]
    cell_metadata = (
        adata.obs[
            output_columns
        ]
        .copy()
    )
    cell_metadata.insert(
        0,
        "cell_id",
        adata.obs_names.astype(str),
    )

    if WRITE_CELL_METADATA_PARQUET:
        cell_metadata.to_parquet(
            paths[
                "metadata"
            ],
            index=False,
        )

    spatial_key = choose_spatial_key(
        adata
    )
    if (
        WRITE_SPATIAL_PARQUET
        and spatial_key
        is not None
    ):
        coordinates = np.asarray(
            adata.obsm[
                spatial_key
            ],
            dtype=np.float32,
        )
        spatial = (
            cell_metadata.copy()
        )
        spatial[
            "spatial_x"
        ] = coordinates[
            :,
            0,
        ]
        spatial[
            "spatial_y"
        ] = coordinates[
            :,
            1,
        ]
        spatial[
            "spatial_source"
        ] = spatial_key
        spatial.to_parquet(
            paths[
                "spatial"
            ],
            index=False,
        )

    # Per-sample count/percentage table.
    sample_counts = (
        adata.obs[
            "prelim_cell_type_primary"
        ]
        .astype(str)
        .value_counts()
        .reindex(
            DETAILED_CELL_TYPE_ORDER,
            fill_value=0,
        )
        .rename_axis(
            "cell_type"
        )
        .rename(
            "n_cells"
        )
        .reset_index()
    )
    sample_counts[
        "percentage"
    ] = (
        100.0
        * sample_counts[
            "n_cells"
        ]
        / max(
            adata.n_obs,
            1,
        )
    )
    sample_counts[
        "sample"
    ] = sample
    sample_counts[
        "patient"
    ] = metadata[
        "patient"
    ]
    sample_counts[
        "cancer_type"
    ] = metadata[
        "cancer_type"
    ]
    sample_counts[
        "biopsy_stage"
    ] = metadata[
        "biopsy_stage"
    ]
    sample_counts.to_csv(
        paths[
            "counts"
        ],
        index=False,
    )

    confidence_counts = pd.concat(
        [
            (
                adata.obs[
                    "prelim_T_confidence_tier"
                ]
                .astype(str)
                .value_counts()
                .rename_axis(
                    "confidence_tier"
                )
                .rename(
                    "n_cells"
                )
                .reset_index()
                .assign(
                    confidence_type="T"
                )
            ),
            (
                adata.obs[
                    "prelim_Treg_confidence_tier"
                ]
                .astype(str)
                .value_counts()
                .rename_axis(
                    "confidence_tier"
                )
                .rename(
                    "n_cells"
                )
                .reset_index()
                .assign(
                    confidence_type="Treg"
                )
            ),
        ],
        ignore_index=True,
    )
    confidence_counts[
        "sample"
    ] = sample
    confidence_counts.to_csv(
        paths[
            "confidence"
        ],
        index=False,
    )

    if WRITE_ANNOTATED_H5AD:
        safe_write_h5ad(
            adata,
            paths[
                "annotated"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )

    if WRITE_PRIMARY_T_ONLY_H5AD:
        primary_mask = bool_obs(
            adata,
            "prelim_T_primary",
        )
        tdata = adata[
            primary_mask
        ].copy()
        tdata.uns[
            "preliminary_cell_type_annotation"
        ][
            "subset_interpretation"
        ] = (
            "Primary rescued T cells, including supported Tregs"
        )
        safe_write_h5ad(
            tdata,
            paths[
                "primary_T"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )
        del tdata
        gc.collect()

    if (
        VALIDATE_WRITTEN_H5AD
        and WRITE_ANNOTATED_H5AD
    ):
        check = ad.read_h5ad(
            paths[
                "annotated"
            ],
            backed="r",
        )
        try:
            if check.n_obs != adata.n_obs:
                raise RuntimeError(
                    f"{sample}: written n_obs changed."
                )
            for column in (
                "prelim_cell_type_primary",
                "prelim_T_confidence_tier",
                "prelim_Treg_confidence_tier",
            ):
                if column not in check.obs.columns:
                    raise RuntimeError(
                        f"{sample}: written object is missing {column}."
                    )
        finally:
            check.file.close()

    summary = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "sample": sample,
        **metadata,
        **annotation_summary,
        "spatial_key": (
            spatial_key
        ),
        "annotated_h5ad": str(
            paths[
                "annotated"
            ]
        )
        if WRITE_ANNOTATED_H5AD
        else None,
        "primary_T_h5ad": str(
            paths[
                "primary_T"
            ]
        )
        if WRITE_PRIMARY_T_ONLY_H5AD
        else None,
        "metadata_parquet": str(
            paths[
                "metadata"
            ]
        )
        if WRITE_CELL_METADATA_PARQUET
        else None,
        "spatial_parquet": str(
            paths[
                "spatial"
            ]
        )
        if (
            WRITE_SPATIAL_PARQUET
            and spatial_key
            is not None
        )
        else None,
    }
    write_json(
        summary,
        paths[
            "summary"
        ],
    )

    print(
        sample,
        annotation_summary[
            "primary_cell_type_counts"
        ],
    )

    del adata
    gc.collect()

    return (
        summary,
        sample_counts,
    )

In [7]:
# ---------------------------------------------------------------------
# Run every sample
# ---------------------------------------------------------------------
results = {}
failures = {}
all_sample_counts = []

for sample in SECTION_NAMES:
    try:
        summary, sample_counts = (
            process_sample(
                sample
            )
        )
        results[
            sample
        ] = summary
        all_sample_counts.append(
            sample_counts
        )

    except Exception as exc:
        failures[
            sample
        ] = (
            f"{type(exc).__name__}: {exc}"
        )
        print(
            f"[FAILED] {sample}: "
            f"{type(exc).__name__}: "
            f"{exc}"
        )
        traceback.print_exc(
            limit=12
        )

        if not CONTINUE_ON_ERROR:
            raise

    finally:
        plt.close(
            "all"
        )
        gc.collect()

write_json(
    results,
    OUTPUT_ROOT
    / "all_sample_preliminary_annotation_results.json",
)
write_json(
    failures,
    OUTPUT_ROOT
    / "all_sample_preliminary_annotation_failures.json",
)

sample_counts_long = (
    pd.concat(
        all_sample_counts,
        ignore_index=True,
    )
    if all_sample_counts
    else pd.DataFrame()
)

sample_counts_long.to_csv(
    TABLE_ROOT
    / "cell_type_counts_and_percentages_by_sample.csv",
    index=False,
)

print(
    "Completed:",
    sorted(
        results
    ),
)
print(
    "Failures:",
    json.dumps(
        failures,
        indent=2,
    ),
)


Preliminary annotation: Screen_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/10_final_inclusive_T_Treg_rescue/Screen_39_21/Screen_39_21_inclusive_rescue_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/Screen_39_21/Screen_39_21_preliminary_cell_type_annotated.h5ad
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/Screen_39_21/Screen_39_21_preliminary_primary_Tcells.h5ad
Screen_39_21 {'Tumor': 2083, 'Fibroblast/stromal': 0, 'Endothelial': 1607, 'Monocyte/macrophage': 0, 'B/plasma': 2014, 'NK': 1549, 'Other/unresolved': 13658, 'Other T': 1314, 'CD4+ T': 325, 'CD8+ T': 308, 'Treg': 92}

Preliminary annotation: C2D15_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrich

In [8]:
# ---------------------------------------------------------------------
# Aggregated tables
# ---------------------------------------------------------------------
if sample_counts_long.empty:
    raise RuntimeError(
        "No completed sample annotations are available for plotting."
    )

sample_counts_long[
    "cancer_type"
] = pd.Categorical(
    sample_counts_long[
        "cancer_type"
    ],
    categories=(
        CANCER_TYPE_ORDER
    ),
    ordered=True,
)
sample_counts_long[
    "biopsy_stage"
] = pd.Categorical(
    sample_counts_long[
        "biopsy_stage"
    ],
    categories=(
        BIOPSY_STAGE_ORDER
    ),
    ordered=True,
)
sample_counts_long[
    "cell_type"
] = pd.Categorical(
    sample_counts_long[
        "cell_type"
    ],
    categories=(
        PLOT_CELL_TYPE_ORDER
    ),
    ordered=True,
)

# Sample-level wide count and percentage tables.
sample_count_wide = (
    sample_counts_long
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        columns="cell_type",
        values="n_cells",
        fill_value=0,
        observed=True,
    )
    .reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )
    .sort_index()
)
sample_percentage_wide = (
    sample_counts_long
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        columns="cell_type",
        values="percentage",
        fill_value=0,
        observed=True,
    )
    .reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )
    .sort_index()
)

sample_count_wide.to_csv(
    TABLE_ROOT
    / "cell_type_counts_by_sample_wide.csv"
)
sample_percentage_wide.to_csv(
    TABLE_ROOT
    / "cell_type_percentages_by_sample_wide.csv"
)

# Patient-level totals across Screen + C2D15.
patient_counts_long = (
    sample_counts_long
    .groupby(
        [
            "cancer_type",
            "patient",
            "cell_type",
        ],
        observed=True,
        as_index=False,
    )[
        "n_cells"
    ]
    .sum()
)
patient_count_wide = (
    patient_counts_long
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
        ],
        columns="cell_type",
        values="n_cells",
        fill_value=0,
        observed=True,
    )
    .reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )
    .sort_index()
)
patient_count_wide.to_csv(
    TABLE_ROOT
    / "cell_type_counts_by_patient_wide.csv"
)

# Screen/C2D15 paired counts within patient.
patient_stage_count_wide = (
    sample_counts_long
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "biopsy_stage",
        ],
        columns="cell_type",
        values="n_cells",
        fill_value=0,
        observed=True,
    )
    .reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )
    .sort_index()
)
patient_stage_count_wide.to_csv(
    TABLE_ROOT
    / "cell_type_counts_by_patient_and_stage_wide.csv"
)

# T-only subtype counts and percentages.
T_subtype_long = (
    sample_counts_long.loc[
        sample_counts_long[
            "cell_type"
        ].astype(str).isin(
            T_SUBTYPE_ORDER
        )
    ]
    .copy()
)
T_subtype_totals = (
    T_subtype_long
    .groupby(
        [
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        observed=True,
    )[
        "n_cells"
    ]
    .transform(
        "sum"
    )
)
T_subtype_long[
    "percentage_within_primary_T"
] = (
    100.0
    * T_subtype_long[
        "n_cells"
    ]
    / T_subtype_totals.replace(
        0,
        np.nan,
    )
)
T_subtype_percentage_wide = (
    T_subtype_long
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        columns="cell_type",
        values=(
            "percentage_within_primary_T"
        ),
        fill_value=0,
        observed=True,
    )
    .reindex(
        columns=T_SUBTYPE_ORDER,
        fill_value=0,
    )
    .sort_index()
)
T_subtype_percentage_wide.to_csv(
    TABLE_ROOT
    / "T_subtype_percentages_within_primary_T_by_sample.csv"
)

display(
    sample_counts_long
)

,cell_type,n_cells,percentage,sample,patient,cancer_type,biopsy_stage
0,Tumor,2083,9.076253,Screen_39_21,patient_39_21,NSCLC,Screen
1,Fibroblast/stromal,0,0.000000,Screen_39_21,patient_39_21,NSCLC,Screen
2,Endothelial,1607,7.002179,Screen_39_21,patient_39_21,NSCLC,Screen
3,Monocyte/macrophage,0,0.000000,Screen_39_21,patient_39_21,NSCLC,Screen
4,B/plasma,2014,8.775599,Screen_39_21,patient_39_21,NSCLC,Screen
...,...,...,...,...,...,...,...
127,Other/unresolved,7077,69.091087,C2D15_23_25,patient_23_25,colon_cancer,C2D15
128,Other T,56,0.546715,C2D15_23_25,patient_23_25,colon_cancer,C2D15
129,CD4+ T,26,0.253832,C2D15_23_25,patient_23_25,colon_cancer,C2D15
130,CD8+ T,29,0.283120,C2D15_23_25,patient_23_25,colon_cancer,C2D15


In [9]:
# ---------------------------------------------------------------------
# Plotting helpers
# ---------------------------------------------------------------------
def ordered_rows_for_cancer(
    frame: pd.DataFrame,
    cancer_type: str,
) -> pd.DataFrame:
    subset = frame.loc[
        frame.index.get_level_values(
            "cancer_type"
        ).astype(str)
        == str(
            cancer_type
        )
    ].copy()

    if subset.empty:
        return subset

    reset = subset.reset_index()
    reset[
        "biopsy_stage"
    ] = pd.Categorical(
        reset[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    reset = reset.sort_values(
        [
            "patient",
            "biopsy_stage",
            "sample",
        ]
    )

    return reset.set_index(
        [
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ]
        if "sample" in reset.columns
        else [
            "cancer_type",
            "patient",
            "biopsy_stage",
        ]
    )


def save_stacked_plot(
    frame: pd.DataFrame,
    *,
    labels: list[str],
    path: Path,
    title: str,
    ylabel: str,
    percentage: bool,
) -> None:
    if frame.empty:
        return

    figure_width = max(
        10,
        0.75
        * len(
            frame
        )
        + 5,
    )
    ax = frame.plot(
        kind="bar",
        stacked=True,
        figsize=(
            figure_width,
            7,
        ),
        width=0.82,
    )
    ax.set_title(
        title
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel
    )
    ax.set_xticklabels(
        labels,
        rotation=35,
        ha="right",
    )
    if percentage:
        ax.set_ylim(
            0,
            100,
        )
    ax.legend(
        title="Cell type",
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=8,
        frameon=False,
    )
    plt.tight_layout()
    plt.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()


def save_patient_total_plot(
    cancer_type: str,
) -> None:
    try:
        subset = patient_count_wide.xs(
            cancer_type,
            level="cancer_type",
        ).copy()
    except KeyError:
        return

    subset = subset.reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )
    labels = subset.index.astype(str).tolist()

    save_stacked_plot(
        subset,
        labels=labels,
        path=(
            FIGURE_ROOT
            / (
                f"{safe_slug(cancer_type)}_"
                "patient_total_cell_type_counts.png"
            )
        ),
        title=(
            f"{cancer_type}: total cell-type counts by patient "
            "(Screen + C2D15)"
        ),
        ylabel="Number of cells",
        percentage=False,
    )


def save_patient_stage_plot(
    cancer_type: str,
) -> None:
    try:
        subset = patient_stage_count_wide.xs(
            cancer_type,
            level="cancer_type",
        ).copy()
    except KeyError:
        return

    reset = subset.reset_index()
    reset[
        "biopsy_stage"
    ] = pd.Categorical(
        reset[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    reset = reset.sort_values(
        [
            "patient",
            "biopsy_stage",
        ]
    )
    labels = [
        f"{patient}\n{stage}"
        for patient, stage in zip(
            reset[
                "patient"
            ],
            reset[
                "biopsy_stage"
            ].astype(str),
        )
    ]
    matrix = (
        reset[
            PLOT_CELL_TYPE_ORDER
        ]
        .copy()
    )

    save_stacked_plot(
        matrix,
        labels=labels,
        path=(
            FIGURE_ROOT
            / (
                f"{safe_slug(cancer_type)}_"
                "patient_paired_Screen_C2D15_cell_type_counts.png"
            )
        ),
        title=(
            f"{cancer_type}: paired Screen/C2D15 cell-type counts"
        ),
        ylabel="Number of cells",
        percentage=False,
    )


def save_sample_percentage_plot(
    cancer_type: str,
) -> None:
    try:
        subset = sample_percentage_wide.xs(
            cancer_type,
            level="cancer_type",
        ).copy()
    except KeyError:
        return

    reset = subset.reset_index()
    reset[
        "biopsy_stage"
    ] = pd.Categorical(
        reset[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    reset = reset.sort_values(
        [
            "patient",
            "biopsy_stage",
            "sample",
        ]
    )
    labels = reset[
        "sample"
    ].astype(str).tolist()
    matrix = (
        reset[
            PLOT_CELL_TYPE_ORDER
        ]
        .copy()
    )

    save_stacked_plot(
        matrix,
        labels=labels,
        path=(
            FIGURE_ROOT
            / (
                f"{safe_slug(cancer_type)}_"
                "sample_cell_type_percentages.png"
            )
        ),
        title=(
            f"{cancer_type}: percentage of each preliminary cell type by sample"
        ),
        ylabel="Percentage of all cells",
        percentage=True,
    )


def save_T_subtype_percentage_plot(
    cancer_type: str,
) -> None:
    try:
        subset = T_subtype_percentage_wide.xs(
            cancer_type,
            level="cancer_type",
        ).copy()
    except KeyError:
        return

    reset = subset.reset_index()
    reset[
        "biopsy_stage"
    ] = pd.Categorical(
        reset[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    reset = reset.sort_values(
        [
            "patient",
            "biopsy_stage",
            "sample",
        ]
    )
    labels = reset[
        "sample"
    ].astype(str).tolist()
    matrix = (
        reset[
            T_SUBTYPE_ORDER
        ]
        .copy()
    )

    save_stacked_plot(
        matrix,
        labels=labels,
        path=(
            FIGURE_ROOT
            / (
                f"{safe_slug(cancer_type)}_"
                "primary_T_subtype_percentages.png"
            )
        ),
        title=(
            f"{cancer_type}: T-cell subtype percentages within primary T cells"
        ),
        ylabel="Percentage of primary T cells",
        percentage=True,
    )


def save_individual_patient_plot(
    cancer_type: str,
    patient: str,
) -> None:
    try:
        subset = patient_stage_count_wide.loc[
            (
                cancer_type,
                patient,
            )
        ].copy()
    except KeyError:
        return

    subset = subset.reindex(
        BIOPSY_STAGE_ORDER
    ).fillna(
        0
    )
    subset = subset.reindex(
        columns=PLOT_CELL_TYPE_ORDER,
        fill_value=0,
    )

    labels = subset.index.astype(str).tolist()
    save_stacked_plot(
        subset,
        labels=labels,
        path=(
            PATIENT_FIGURE_ROOT
            / (
                f"{safe_slug(cancer_type)}_"
                f"{safe_slug(patient)}_"
                "Screen_vs_C2D15_counts.png"
            )
        ),
        title=(
            f"{cancer_type} — {patient}: Screen versus C2D15"
        ),
        ylabel="Number of cells",
        percentage=False,
    )

In [10]:
# ---------------------------------------------------------------------
# Generate all requested plots
# ---------------------------------------------------------------------
for cancer_type in CANCER_TYPE_ORDER:
    save_patient_total_plot(
        cancer_type
    )
    save_patient_stage_plot(
        cancer_type
    )
    save_sample_percentage_plot(
        cancer_type
    )
    save_T_subtype_percentage_plot(
        cancer_type
    )

    patients = sorted(
        sample_counts_long.loc[
            sample_counts_long[
                "cancer_type"
            ].astype(str)
            == str(
                cancer_type
            ),
            "patient",
        ]
        .astype(str)
        .unique()
    )
    for patient in patients:
        save_individual_patient_plot(
            cancer_type,
            patient,
        )

print(
    "Figures saved under:",
    FIGURE_ROOT,
)

Figures saved under: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/figures


In [11]:
# ---------------------------------------------------------------------
# Cross-sample confidence summaries and manifest
# ---------------------------------------------------------------------
sample_summary = pd.DataFrame(
    [
        result
        for result in (
            results.values()
        )
    ]
)
sample_summary.to_csv(
    TABLE_ROOT
    / "preliminary_annotation_summary_by_sample.csv",
    index=False,
)

# Cell-type percentage sanity check.
percentage_check = (
    sample_counts_long
    .groupby(
        [
            "sample",
        ],
        observed=True,
    )[
        "percentage"
    ]
    .sum()
    .rename(
        "percentage_sum"
    )
    .reset_index()
)
percentage_check[
    "is_close_to_100"
] = np.isclose(
    percentage_check[
        "percentage_sum"
    ],
    100.0,
    rtol=0,
    atol=1e-6,
)
percentage_check.to_csv(
    TABLE_ROOT
    / "sample_percentage_validation.csv",
    index=False,
)

if not percentage_check[
    "is_close_to_100"
].all():
    raise RuntimeError(
        "At least one sample's cell-type percentages do not sum to 100%."
    )

manifest = {
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "method": (
        "Step 10 primary T + supported Treg "
        "combined with Step 08 broad lineage"
    ),
    "primary_T_column": (
        PRIMARY_T_COLUMN
    ),
    "primary_Treg_column": (
        PRIMARY_TREG_COLUMN
    ),
    "primary_cell_type_column": (
        "prelim_cell_type_primary"
    ),
    "exploratory_cell_type_column": (
        "prelim_cell_type_exploratory"
    ),
    "T_confidence_column": (
        "prelim_T_confidence_tier"
    ),
    "Treg_confidence_column": (
        "prelim_Treg_confidence_tier"
    ),
    "n_samples_completed": int(
        len(
            results
        )
    ),
    "n_samples_failed": int(
        len(
            failures
        )
    ),
    "annotated_sample_summary": str(
        TABLE_ROOT
        / "preliminary_annotation_summary_by_sample.csv"
    ),
    "sample_counts_and_percentages": str(
        TABLE_ROOT
        / "cell_type_counts_and_percentages_by_sample.csv"
    ),
    "patient_counts": str(
        TABLE_ROOT
        / "cell_type_counts_by_patient_wide.csv"
    ),
    "sample_percentages": str(
        TABLE_ROOT
        / "cell_type_percentages_by_sample_wide.csv"
    ),
    "T_subtype_percentages": str(
        TABLE_ROOT
        / "T_subtype_percentages_within_primary_T_by_sample.csv"
    ),
    "figure_root": str(
        FIGURE_ROOT
    ),
    "failures": failures,
}
write_json(
    manifest,
    OUTPUT_ROOT
    / "preliminary_annotation_manifest.json",
)

display(
    sample_summary
)
display(
    sample_percentage_wide
)
display(
    T_subtype_percentage_wide
)

,pipeline_version,sample,patient,cancer_type,biopsy_stage,n_cells,n_strict_T,n_primary_T,n_exploratory_T,n_Treg_high_confidence,n_Treg_supported,n_Treg_exploratory,primary_cell_type_counts,spatial_key,annotated_h5ad,primary_T_h5ad,metadata_parquet,spatial_parquet
0,2026-08-02-preliminary-cell-type-annotation-v1,Screen_39_21,patient_39_21,NSCLC,Screen,22950,1114,2039,2687,7,92,117,"{'Tumor': 2083, 'Fibroblast/stromal': 0, 'Endo...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
1,2026-08-02-preliminary-cell-type-annotation-v1,C2D15_39_21,patient_39_21,NSCLC,C2D15,10822,1079,1172,1339,1,22,25,"{'Tumor': 1017, 'Fibroblast/stromal': 0, 'Endo...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
2,2026-08-02-preliminary-cell-type-annotation-v1,Screen_17_26,patient_17_26,NSCLC,Screen,47896,0,754,1350,1,31,57,"{'Tumor': 0, 'Fibroblast/stromal': 4679, 'Endo...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
3,2026-08-02-preliminary-cell-type-annotation-v1,C2D15_17_26,patient_17_26,NSCLC,C2D15,87913,0,891,1627,3,43,66,"{'Tumor': 729, 'Fibroblast/stromal': 8611, 'En...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
4,2026-08-02-preliminary-cell-type-annotation-v1,Screen_18_23,patient_18_23,melanoma,Screen,72386,147,923,1343,0,0,0,"{'Tumor': 4193, 'Fibroblast/stromal': 189, 'En...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
5,2026-08-02-preliminary-cell-type-annotation-v1,C2D15_18_23,patient_18_23,melanoma,C2D15,18594,34,821,1268,0,6,6,"{'Tumor': 1146, 'Fibroblast/stromal': 775, 'En...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
6,2026-08-02-preliminary-cell-type-annotation-v1,Screen_16_22,patient_16_22,melanoma,Screen,70438,76,1244,2369,0,2,4,"{'Tumor': 0, 'Fibroblast/stromal': 0, 'Endothe...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
7,2026-08-02-preliminary-cell-type-annotation-v1,C2D15_16_22,patient_16_22,melanoma,C2D15,76633,171,2554,4214,9,128,167,"{'Tumor': 0, 'Fibroblast/stromal': 0, 'Endothe...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
8,2026-08-02-preliminary-cell-type-annotation-v1,Screen_30_16,patient_30_16,melanoma,Screen,66977,1690,4148,6096,19,205,235,"{'Tumor': 0, 'Fibroblast/stromal': 5826, 'Endo...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
9,2026-08-02-preliminary-cell-type-annotation-v1,C2D15_30_16,patient_30_16,melanoma,C2D15,351799,0,437,1241,0,20,43,"{'Tumor': 3417, 'Fibroblast/stromal': 28133, '...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj.

cell_type                                                Tumor  \
cancer_type  patient       sample       biopsy_stage             
melanoma     patient_16_22 C2D15_16_22  C2D15         0.000000   
                           Screen_16_22 Screen        0.000000   
             patient_18_23 C2D15_18_23  C2D15         6.163278   
                           Screen_18_23 Screen        5.792557   
             patient_30_16 C2D15_30_16  C2D15         0.971293   
                           Screen_30_16 Screen        0.000000   
NSCLC        patient_17_26 C2D15_17_26  C2D15         0.829229   
                           Screen_17_26 Screen        0.000000   
             patient_39_21 C2D15_39_21  C2D15         9.397524   
                           Screen_39_21 Screen        9.076253   
colon_cancer patient_23_25 C2D15_23_25  C2D15         2.216148   
                           Screen_23_25 Screen        3.518777   

cell_type                                             Fibroblast/stromal  \
cancer_type  patient       sample       biopsy_stage                       
melanoma     patient_16_22 C2D15_16_22  C2D15                   0.000000   
                           Screen_16_22 Screen                  0.000000   
             patient_18_23 C2D15_18_23  C2D15                   4.168011   
                           Screen_18_23 Screen                  0.261100   
             patient_30_16 C2D15_30_16  C2D15                   7.996896   
                           Screen_30_16 Screen                  8.698508   
NSCLC        patient_17_26 C2D15_17_26  C2D15                   9.794911   
                           Screen_17_26 Screen                  9.769083   
             patient_39_21 C2D15_39_21  C2D15                   0.000000   
                           Screen_39_21 Screen                  0.000000   
colon_cancer patient_23_25 C2D15_23_25  C2D15                   0.000000   
                           Screen_23_25 Screen                  6.385785   

cell_type                                             Endothelial  \
cancer_type  patient       sample       biopsy_stage                
melanoma     patient_16_22 C2D15_16_22  C2D15            4.932601   
                           Screen_16_22 Screen           6.805985   
             patient_18_23 C2D15_18_23  C2D15            8.529633   
                           Screen_18_23 Screen           2.604095   
             patient_30_16 C2D15_30_16  C2D15            0.131609   
                           Screen_30_16 Screen           8.540245   
NSCLC        patient_17_26 C2D15_17_26  C2D15            0.000000   
                           Screen_17_26 Screen           0.000000   
             patient_39_21 C2D15_39_21  C2D15            9.868786   
                           Screen_39_21 Screen           7.002179   
colon_cancer patient_23_25 C2D15_23_25  C2D15            8.230011   
                           Screen_23_25 Screen           5.563315   

cell_type                                             Monocyte/macrophage  \
cancer_type  patient       sample       biopsy_stage                        
melanoma     patient_16_22 C2D15_16_22  C2D15                    7.722522   
                           Screen_16_22 Screen                   7.285840   
             patient_18_23 C2D15_18_23  C2D15                    7.136711   
                           Screen_18_23 Screen                   8.554140   
             patient_30_16 C2D15_30_16  C2D15                    9.233113   
                           Screen_30_16 Screen                   0.000000   
NSCLC        patient_17_26 C2D15_17_26  C2D15                    2.352326   
                           Screen_17_26 Screen                   4.710205   
             patient_39_21 C2D15_39_21  C2D15                    0.055443   
                           Screen_39_21 Screen                   0.000000   
colon_cancer patient_23_25 C2D15_23_25  C2D15                    9.313678   
                           Screen_23_25 Sc

cell_type                                               Other T     CD4+ T  \
cancer_type  patient       sample       biopsy_stage                         
melanoma     patient_16_22 C2D15_16_22  C2D15         68.363352  13.743148   
                           Screen_16_22 Screen        79.983923  10.128617   
             patient_18_23 C2D15_18_23  C2D15         52.009744  23.264312   
                           Screen_18_23 Screen        66.305525  18.093174   
             patient_30_16 C2D15_30_16  C2D15         62.471396  19.679634   
                           Screen_30_16 Screen        67.695275  13.162970   
NSCLC        patient_17_26 C2D15_17_26  C2D15         72.278339  11.447811   
                           Screen_17_26 Screen        71.352785  12.864721   
             patient_39_21 C2D15_39_21  C2D15         85.665529   6.569966   
                           Screen_39_21 Screen        64.443355  15.939186   
colon_cancer patient_23_25 C2D15_23_25  C2D15         48.695652  22.608696   
                           Screen_23_25 Screen        74.747475  11.111111   

cell_type                                                CD8+ T      Treg  
cancer_type  patient       sample       biopsy_stage                       
melanoma     patient_16_22 C2D15_16_22  C2D15         12.881754  5.011746  
                           Screen_16_22 Screen         9.726688  0.160772  
             patient_18_23 C2D15_18_23  C2D15         23.995128  0.730816  
                           Screen_18_23 Screen        15.601300  0.000000  
             patient_30_16 C2D15_30_16  C2D15         13.272311  4.576659  
                           Screen_30_16 Screen        14.199614  4.942141  
NSCLC        patient_17_26 C2D15_17_26  C2D15         11.447811  4.826038  
                           Screen_17_26 Screen        11.671088  4.111406  
             patient_39_21 C2D15_39_21  C2D15          5.887372  1.877133  
                           Screen_39_21 Screen        15.105444  4.512016  
colon_cancer patient_23_25 C2D15_23_25  C2D15         25.217391  3.478261  
                           Screen_23_25 Screen         9.090909  5.050505

# Recommended downstream columns

## Main preliminary annotation

```python
adata.obs["prelim_cell_type_primary"]
```

This uses:

```text
primary T cells
supported Tregs
CD4/CD8 primary subtype calls
fallback tumor/endothelial/myeloid/fibroblast/B/NK calls
```

## Sensitivity annotation

```python
adata.obs["prelim_cell_type_exploratory"]
```

## Confidence records

```python
adata.obs["prelim_T_confidence_tier"]
adata.obs["prelim_T_confidence_rank"]

adata.obs["prelim_Treg_confidence_tier"]
adata.obs["prelim_Treg_confidence_rank"]
```

## Boolean convenience columns

```python
prelim_T_primary
prelim_T_exploratory

prelim_CD4_T
prelim_CD8_T
prelim_Treg

prelim_Treg_high_confidence
prelim_Treg_supported
prelim_Treg_exploratory
```

## Spatial/mixing audit

```python
rescue_T_mixing_status
rescue_T_percentile_minus_max_nonT
```

## DEG recommendation

For the preliminary Screen-versus-C2D15 analysis:

```text
all primary T:
    prelim_T_primary

CD4:
    prelim_CD4_T

CD8:
    prelim_CD8_T

Treg primary exploratory:
    prelim_Treg_supported

Treg specificity analysis:
    prelim_Treg_high_confidence
```

Use patient/sample as the biological replicate. Individual cells should not be
treated as independent replicates.